# 🧠 Brain Tumor Segmentation — Improved Training Pipeline

## Vấn đề với U-Net thuần (Dice ~0.57)
| Vấn đề | Nguyên nhân | Giải pháp |
|---|---|---|
| Encoder yếu | Random init, học từ đầu | EfficientNet-B4 pretrained ImageNet |
| Loss chưa tối | BCE+Dice cơ bản | Focal-Tversky + Dice + BCE combo |
| Augmentation thiếu | Flip+Rotate đơn giản | Albumentations đầy đủ |
| LR schedule kém | CosineAnnealing đơn | Warmup + CosineAnnealing |
| Không dùng TTA | Predict 1 lần | Test-Time Augmentation x8 |
| Threshold cứng 0.5 | Fixed threshold | Tìm optimal threshold trên val |

## Kết quả kỳ vọng
- **U-Net thuần**: Dice ~0.57  
- **U-Net++ EfficientNet-B4**: Dice ~0.87-0.92


In [ ]:
# Cài đặt (bỏ comment nếu cần)
# !pip install segmentation-models-pytorch timm albumentations -q
# !pip install torch torchvision opencv-python-headless scikit-image scipy tqdm -q
# !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation -q
# !unzip -q lgg-mri-segmentation.zip -d dataset/

In [ ]:
import os, sys, random, glob, json, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, '../backend')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ════════════════════════════════════════════════
# CONFIG v2 — các thay đổi chính để tăng Dice
# ════════════════════════════════════════════════
CFG = {
    # Model
    'arch'    : 'unetplusplus',
    'encoder' : 'efficientnet-b2',   # ← Đổi từ b4→b2: nhỏ hơn, ít overfit hơn
                                     #   Hoặc giữ b4 nhưng thêm dropout bên dưới

    # Data
    'data_root'  : '../dataset/BrainMRI',
    'img_size'   : 384,              # ← 256→384: giữ detail khối u nhỏ (giảm batch nếu OOM)
    'val_split'  : 0.15,
    'num_workers': 2,
    'no_tumor_ratio': 0.0,           # ← MỚI: 0.0 = bỏ hoàn toàn no-tumor images

    # Training
    'epochs'      : 80,              # ← 60→80: cho model học thêm
    'batch_size'  : 6,               # ← 8→6: do img_size tăng
    'lr'          : 2e-4,
    'encoder_lr'  : 2e-5,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,              # ← 3→5
    'amp'         : True,

    # Loss weights
    'w_dice'  : 0.5,
    'w_focal' : 0.3,
    'w_bce'   : 0.2,
    'pos_weight': 15.0,              # ← MỚI: weight cho positive class (tumor pixel)
                                     #   = approx (neg_pixels / pos_pixels) ≈ 10-20

    # Focal Tversky
    'tversky_alpha': 0.7,
    'tversky_beta' : 0.3,
    'focal_gamma'  : 0.75,

    # Dropout in decoder (tránh overfit)
    'decoder_dropout': 0.3,          # ← MỚI: 0.0 = tắt, 0.3 = khuyến nghị

    # TTA at inference
    'tta': True,

    # Save
    'save_dir': '../backend/weights',
    'save_name': 'unet_best.pth',
}

os.makedirs(CFG['save_dir'], exist_ok=True)
print('Config v2 loaded ✓')
print(f'  Image size     : {CFG["img_size"]}×{CFG["img_size"]}')
print(f'  No-tumor ratio : {CFG["no_tumor_ratio"]}')
print(f'  pos_weight     : {CFG["pos_weight"]}')
print(f'  Decoder dropout: {CFG["decoder_dropout"]}')


In [ ]:
# ════════════════════════════════════════════════
# DATASET + AUGMENTATION v2
# ════════════════════════════════════════════════
def get_transforms(mode='train', size=384):
    if mode == 'train':
        return A.Compose([
            A.Resize(size, size),
            # Spatial
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2,
                               rotate_limit=30, border_mode=cv2.BORDER_REFLECT, p=0.6),
            A.ElasticTransform(alpha=120, sigma=120*0.05,
                               alpha_affine=120*0.03, p=0.3),
            A.GridDistortion(p=0.2),
            # Intensity
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1),
                A.RandomGamma(gamma_limit=(70, 130), p=1),
                A.CLAHE(clip_limit=4.0, p=1),
            ], p=0.6),
            A.GaussianBlur(blur_limit=(3, 7), p=0.2),
            A.GaussNoise(var_limit=(10, 50), p=0.2),
            A.CoarseDropout(max_holes=4, max_height=32, max_width=32,
                            min_holes=1, fill_value=0, p=0.2),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(size, size),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])


class BrainMRIDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.imgs      = img_paths
        self.masks     = mask_paths
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.imgs[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']
            mask = aug['mask'].float().unsqueeze(0)
        return img, mask


# ── Load & filter data ──────────────────────────────────────────────────
all_imgs  = sorted(glob.glob(os.path.join(CFG['data_root'], '**', '*[!_mask].tif'), recursive=True))
all_masks = sorted(glob.glob(os.path.join(CFG['data_root'], '**', '*_mask.tif'),    recursive=True))
print(f'Total images : {len(all_imgs)} | Total masks: {len(all_masks)}')

# Phân loại has_tumor / no_tumor
pairs = list(zip(all_imgs, all_masks))
random.seed(42); random.shuffle(pairs)

has_tumor, no_tumor = [], []
print('Scanning masks...', end='', flush=True)
for i, (img_p, msk_p) in enumerate(pairs):
    m = cv2.imread(msk_p, cv2.IMREAD_GRAYSCALE)
    if m is None: continue
    if m.max() > 0: has_tumor.append((img_p, msk_p))
    else:           no_tumor.append((img_p, msk_p))
    if (i+1) % 100 == 0: print('.', end='', flush=True)
print(f' done')
print(f'Has tumor: {len(has_tumor)} | No tumor: {len(no_tumor)}')

# Giữ no-tumor theo tỉ lệ trong CFG
keep_no   = int(len(has_tumor) * CFG['no_tumor_ratio'])
balanced  = has_tumor + no_tumor[:keep_no]
random.shuffle(balanced)

n_val       = int(len(balanced) * CFG['val_split'])
train_pairs = balanced[n_val:]
val_pairs   = balanced[:n_val]
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} (no-tumor kept: {keep_no})')

train_imgs, train_masks = zip(*train_pairs)
val_imgs,   val_masks   = zip(*val_pairs)

train_ds = BrainMRIDataset(list(train_imgs), list(train_masks),
                            transform=get_transforms('train', CFG['img_size']))
val_ds   = BrainMRIDataset(list(val_imgs),   list(val_masks),
                            transform=get_transforms('val',   CFG['img_size']))

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


In [ ]:
# ════════════════════════════════════════════════
# LOSS FUNCTIONS v2 — pos_weight cho BCELoss
# ════════════════════════════════════════════════

# pos_weight: penalize False Negative (bỏ sót khối u) mạnh hơn
POS_WEIGHT = torch.tensor([CFG['pos_weight']], device=DEVICE)

def dice_loss(pred, target, smooth=1.0):
    pred   = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    inter  = (pred * target).sum()
    return 1 - (2*inter + smooth) / (pred.sum() + target.sum() + smooth)

def focal_tversky_loss(pred, target,
                       alpha=CFG['tversky_alpha'],
                       beta=CFG['tversky_beta'],
                       gamma=CFG['focal_gamma'],
                       smooth=1.0):
    pred   = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    TP     = (pred * target).sum()
    FP     = ((1-target) * pred).sum()
    FN     = (target * (1-pred)).sum()
    tversky = (TP + smooth) / (TP + alpha*FN + beta*FP + smooth)
    return (1 - tversky) ** gamma

def bce_loss(pred, target):
    # Dùng pos_weight: mỗi pixel tumor bị tính nặng gấp pos_weight lần
    return F.binary_cross_entropy_with_logits(pred, target, pos_weight=POS_WEIGHT)

def combined_loss(pred, target):
    return (CFG['w_dice']  * dice_loss(pred, target) +
            CFG['w_focal'] * focal_tversky_loss(pred, target) +
            CFG['w_bce']   * bce_loss(pred, target))

def dice_score(pred, target, thresh=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > thresh).float().view(-1)
    target = target.view(-1)
    inter  = (pred * target).sum()
    return ((2*inter + smooth) / (pred.sum() + target.sum() + smooth)).item()

def iou_score(pred, target, thresh=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > thresh).float().view(-1)
    target = target.view(-1)
    inter  = (pred * target).sum()
    union  = pred.sum() + target.sum() - inter
    return ((inter + smooth) / (union + smooth)).item()

print('Loss functions v2 defined ✓')
print(f'  pos_weight = {CFG["pos_weight"]} (tumor pixel x{CFG["pos_weight"]} nặng hơn)')


In [ ]:
# ════════════════════════════════════════════════
# MODEL v2 — thêm Dropout vào decoder
# ════════════════════════════════════════════════
import segmentation_models_pytorch as smp

model = smp.create_model(
    CFG['arch'],
    encoder_name    = CFG['encoder'],
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 1,
    activation      = None,
    decoder_use_batchnorm = True,
)

# Thêm Dropout vào decoder để tránh overfit
# (SMP không expose trực tiếp — inject vào từng block)
if CFG['decoder_dropout'] > 0:
    import torch.nn as nn
    injected = 0
    for name, module in model.named_modules():
        # Inject sau BatchNorm2d cuối trong mỗi decoder block
        if isinstance(module, nn.Sequential):
            children = list(module.named_children())
            for i, (child_name, child) in enumerate(children):
                if isinstance(child, nn.ReLU) and i > 0:
                    # Wrap ReLU + Dropout
                    pass  # SMP blocks have fixed structure, use hook instead
    # Simpler: register forward hooks để dropout feature maps
    def make_dropout_hook(p):
        drop = nn.Dropout2d(p=p)
        def hook(m, inp, out):
            if m.training:
                return drop(out)
        return hook

    # Apply dropout to last 2 decoder blocks
    decoder_blocks = list(model.decoder.blocks)
    for blk in decoder_blocks[-2:]:
        blk.conv2.register_forward_hook(make_dropout_hook(CFG['decoder_dropout']))
        injected += 1
    print(f'  Dropout {CFG["decoder_dropout"]} injected vào {injected} decoder blocks')

model = model.to(DEVICE)
total  = sum(p.numel() for p in model.parameters())
print(f'Architecture : {CFG["arch"]} + {CFG["encoder"]}')
print(f'Image size   : {CFG["img_size"]}×{CFG["img_size"]}')
print(f'Total params : {total/1e6:.1f}M')


In [ ]:
# ════════════════════════════════════════════════
# OPTIMIZER v2 — warmup + cosine + ReduceLROnPlateau
# ════════════════════════════════════════════════
encoder_params = list(model.encoder.parameters())
decoder_params = [p for n,p in model.named_parameters() if not n.startswith('encoder')]

optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': CFG['encoder_lr']},
    {'params': decoder_params, 'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

def warmup_cosine(epoch, warmup=CFG['warmup_epochs'], total=CFG['epochs']):
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return max(0.05, 0.5 * (1 + np.cos(np.pi * progress)))  # không về 0

# Cosine + không về 0 (min_lr = 5% của LR gốc)
scheduler_cosine = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine)

# ReduceLROnPlateau: nếu val_dice không tăng sau 8 epoch → giảm LR ×0.5
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7, verbose=True)

scaler = GradScaler(enabled=CFG['amp'] and torch.cuda.is_available())

# Early stopping
EARLY_STOP_PATIENCE = 15

print('Optimizer v2 configured ✓')
print(f'  Warmup: {CFG["warmup_epochs"]} epochs')
print(f'  Cosine T_max: {CFG["epochs"]} epochs (min LR = 5%)')
print(f'  ReduceLROnPlateau: patience=8, factor=0.5')
print(f'  Early stopping: patience={EARLY_STOP_PATIENCE}')


In [ ]:
# ════════════════════════════════════════════════
# TRAINING LOOP v2 — early stopping + dual scheduler
# ════════════════════════════════════════════════
best_dice     = 0.0
best_thresh   = 0.5
no_improve    = 0
history       = {'train_loss':[], 'val_loss':[], 'val_dice':[], 'val_iou':[], 'lr':[]}

for epoch in range(1, CFG['epochs'] + 1):

    # ── TRAIN ──────────────────────────────────────────────────────────
    model.train()
    t_loss = 0.0
    bar = tqdm(train_loader, desc=f'Epoch {epoch:3d}/{CFG["epochs"]} [Train]', ncols=95)
    for imgs, masks in bar:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with autocast(enabled=CFG['amp'] and torch.cuda.is_available()):
            preds = model(imgs)
            loss  = combined_loss(preds, masks)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')
    t_loss /= len(train_loader)

    # ── VALIDATE ────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_dice, v_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds  = model(imgs)
            v_loss += combined_loss(preds, masks).item()
            v_dice += dice_score(preds, masks)
            v_iou  += iou_score(preds, masks)
    v_loss /= len(val_loader)
    v_dice /= len(val_loader)
    v_iou  /= len(val_loader)

    # ── SCHEDULERS ──────────────────────────────────────────────────────
    scheduler_cosine.step()
    scheduler_plateau.step(v_dice)          # dùng val_dice làm metric

    lr_now = optimizer.param_groups[1]['lr']
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_dice'].append(v_dice)
    history['val_iou'].append(v_iou)
    history['lr'].append(lr_now)

    gap = v_loss - t_loss
    print(f'Epoch {epoch:3d} | Loss: {t_loss:.4f} | VLoss: {v_loss:.4f} '
          f'| Gap: {gap:+.4f} | Dice: {v_dice:.4f} | IoU: {v_iou:.4f} | LR: {lr_now:.2e}')

    # ── CHECKPOINT ──────────────────────────────────────────────────────
    if v_dice > best_dice:
        best_dice = v_dice
        no_improve = 0
        torch.save({
            'epoch'           : epoch,
            'arch'            : CFG['arch'],
            'encoder'         : CFG['encoder'],
            'model_state_dict': model.state_dict(),
            'best_dice'       : best_dice,
            'threshold'       : best_thresh,
        }, f'{CFG["save_dir"]}/{CFG["save_name"]}')
        print(f'  ✓ Saved best model  Dice={best_dice:.4f}')
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f'\n⏹ Early stopping tại epoch {epoch} (no improve for {EARLY_STOP_PATIENCE} epochs)')
            break

print(f'\n✅ Training complete  Best Dice: {best_dice:.4f}')


In [ ]:
# ════════════════════════════════════════════════
# TÌM OPTIMAL THRESHOLD TRÊN VAL SET
# ════════════════════════════════════════════════
# Load best weights
ckpt = torch.load(f'{CFG["save_dir"]}/{CFG["save_name"]}', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

thresholds = np.arange(0.3, 0.75, 0.025)
thresh_dice = []

with torch.no_grad():
    all_preds, all_masks_np = [], []
    for imgs, masks in tqdm(val_loader, desc='Collecting preds'):
        imgs = imgs.to(DEVICE)
        preds = torch.sigmoid(model(imgs)).cpu().numpy()
        all_preds.append(preds)
        all_masks_np.append(masks.numpy())

all_preds   = np.concatenate(all_preds,    axis=0)
all_masks_np= np.concatenate(all_masks_np, axis=0)

for t in thresholds:
    bin_pred = (all_preds > t).astype(np.float32)
    inter = (bin_pred * all_masks_np).sum()
    dice  = (2*inter + 1) / (bin_pred.sum() + all_masks_np.sum() + 1)
    thresh_dice.append(dice)

best_thresh = thresholds[np.argmax(thresh_dice)]
best_t_dice = max(thresh_dice)
print(f'Optimal threshold : {best_thresh:.3f}')
print(f'Dice at optimal t : {best_t_dice:.4f}')

plt.figure(figsize=(8, 4), facecolor='#060d14')
ax = plt.gca(); ax.set_facecolor('#0a1520')
plt.plot(thresholds, thresh_dice, color='#00c8ff', linewidth=2)
plt.axvline(best_thresh, color='#00ff88', linestyle='--',
            label=f'Best t={best_thresh:.3f} (Dice={best_t_dice:.4f})')
plt.xlabel('Threshold', color='#aaa'); plt.ylabel('Dice Score', color='#aaa')
plt.title('Threshold Search', color='#e8f4f8'); plt.legend()
plt.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig('../outputs/threshold_search.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ════════════════════════════════════════════════
# TEST-TIME AUGMENTATION (TTA)
# ════════════════════════════════════════════════
def tta_predict(model, img_tensor, n_aug=8):
    """
    Trung bình 8 lần predict với các augmentation khác nhau.
    img_tensor: 1×C×H×W
    """
    preds = []
    augs  = [
        lambda x: x,                             # original
        lambda x: torch.flip(x, [3]),            # H-flip
        lambda x: torch.flip(x, [2]),            # V-flip
        lambda x: torch.rot90(x, 1, [2, 3]),     # rot 90
        lambda x: torch.rot90(x, 2, [2, 3]),     # rot 180
        lambda x: torch.rot90(x, 3, [2, 3]),     # rot 270
        lambda x: torch.flip(torch.rot90(x, 1, [2,3]), [3]),   # rot90 + H-flip
        lambda x: torch.flip(torch.rot90(x, 3, [2,3]), [3]),   # rot270 + H-flip
    ]
    de_augs = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.flip(x, [2]),
        lambda x: torch.rot90(x, 3, [2, 3]),
        lambda x: torch.rot90(x, 2, [2, 3]),
        lambda x: torch.rot90(x, 1, [2, 3]),
        lambda x: torch.flip(torch.rot90(x, 3, [2,3]), [3]),
        lambda x: torch.flip(torch.rot90(x, 1, [2,3]), [3]),
    ]
    model.eval()
    with torch.no_grad():
        for aug, de_aug in zip(augs[:n_aug], de_augs[:n_aug]):
            x_aug  = aug(img_tensor)
            logits = model(x_aug)
            prob   = torch.sigmoid(logits)
            preds.append(de_aug(prob))
    return torch.stack(preds).mean(0)  # 1×1×H×W


# Đánh giá TTA trên val set
print('Đánh giá TTA...')
tta_scores = []
model.eval()
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='TTA Val'):
        imgs = imgs.to(DEVICE)
        prob = tta_predict(model, imgs)
        pred = (prob > best_thresh).float().cpu()
        inter = (pred * masks).sum()
        dice  = (2*inter + 1) / (pred.sum() + masks.sum() + 1)
        tta_scores.append(dice.item())

print(f'TTA Val Dice : {np.mean(tta_scores):.4f}')
print(f'Improvement  : +{np.mean(tta_scores) - best_t_dice:.4f}')

In [ ]:
# ════════════════════════════════════════════════
# VISUALIZE — xem kết quả trên val samples
# ════════════════════════════════════════════════
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denorm(t):
    img = t.permute(1,2,0).numpy()
    img = (img * STD + MEAN).clip(0,1)
    return (img * 255).astype(np.uint8)

model.eval()
fig, axes = plt.subplots(4, 5, figsize=(18, 14), facecolor='#060d14')
plt.suptitle(f'Val Predictions (thresh={best_thresh:.2f})',
             color='#e8f4f8', fontsize=14, y=1.01)

val_iter = iter(val_loader)
imgs_b, masks_b = next(val_iter)

with torch.no_grad():
    if CFG['tta']:
        probs = tta_predict(model, imgs_b.to(DEVICE)).cpu()
    else:
        probs = torch.sigmoid(model(imgs_b.to(DEVICE))).cpu()

for i in range(min(4, len(imgs_b))):
    img_show  = denorm(imgs_b[i])
    mask_show = masks_b[i,0].numpy()
    pred_show = (probs[i,0].numpy() > best_thresh).astype(np.uint8)

    # Overlay
    overlay = img_show.copy()
    overlay[pred_show==1] = [0, 220, 120]
    overlay = cv2.addWeighted(img_show, 0.55, overlay, 0.45, 0)

    diff = np.zeros((*pred_show.shape, 3), dtype=np.uint8)
    diff[(pred_show==1) & (mask_show==1)] = [0, 255, 100]   # TP green
    diff[(pred_show==1) & (mask_show==0)] = [255, 80, 80]   # FP red
    diff[(pred_show==0) & (mask_show==1)] = [255, 220, 0]   # FN yellow

    inter = (pred_show * mask_show).sum()
    d = (2*inter+1)/(pred_show.sum()+mask_show.sum()+1)

    titles = [f'MRI (Dice={d:.3f})', 'GT Mask', 'Pred Overlay', 'TP/FP/FN', 'Prob Map']
    imgs_show_list = [
        img_show, mask_show*255, overlay, diff,
        (probs[i,0].numpy()*255).astype(np.uint8)
    ]
    cmaps = [None, 'gray', None, None, 'jet']

    for j, (ax, title, im, cm) in enumerate(zip(axes[i], titles, imgs_show_list, cmaps)):
        ax.set_facecolor('#0a1520')
        ax.imshow(im, cmap=cm)
        ax.set_title(title, color='#e8f4f8', fontsize=9)
        ax.axis('off')

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/val_predictions.png', dpi=150, bbox_inches='tight',
            facecolor='#060d14')
plt.show()
print('Saved to ../outputs/val_predictions.png')

In [ ]:
# ════════════════════════════════════════════════
# TRAINING HISTORY
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#060d14')
epochs_x = range(1, len(history['train_loss'])+1)

plots = [
    (axes[0], 'Loss', [
        (history['train_loss'], '#00c8ff', 'Train Loss'),
        (history['val_loss'],   '#ff6b35', 'Val Loss'),
    ]),
    (axes[1], 'Metrics', [
        (history['val_dice'], '#00ff88', f'Val Dice (best={best_dice:.4f})'),
        (history['val_iou'],  '#ffd60a', 'Val IoU'),
    ]),
    (axes[2], 'Learning Rate', [
        (history['lr'], '#c77dff', 'LR (decoder)'),
    ]),
]

for ax, title, lines in plots:
    ax.set_facecolor('#0a1520')
    ax.spines[:].set_color('#1a2a3a')
    ax.tick_params(colors='#888')
    ax.set_title(title, color='#e8f4f8', fontsize=12)
    for data, color, label in lines:
        ax.plot(epochs_x, data, color=color, linewidth=2, label=label)
    ax.legend(facecolor='#060d14', labelcolor='#ccc', fontsize=9)
    ax.set_xlabel('Epoch', color='#888')

plt.tight_layout()
plt.savefig('../outputs/training_history.png', dpi=150,
            bbox_inches='tight', facecolor='#060d14')
plt.show()
print('Saved to ../outputs/training_history.png')

In [ ]:
# ════════════════════════════════════════════════
# FINAL EVALUATION SUMMARY
# ════════════════════════════════════════════════
from sklearn.metrics import precision_score, recall_score, f1_score

model.eval()
all_pred_flat = []
all_mask_flat = []

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Final eval'):
        imgs = imgs.to(DEVICE)
        if CFG['tta']:
            prob = tta_predict(model, imgs).cpu()
        else:
            prob = torch.sigmoid(model(imgs)).cpu()
        pred = (prob > best_thresh).long().numpy().flatten()
        mask = masks.long().numpy().flatten()
        all_pred_flat.extend(pred.tolist())
        all_mask_flat.extend(mask.tolist())

all_pred_flat = np.array(all_pred_flat)
all_mask_flat = np.array(all_mask_flat)

precision = precision_score(all_mask_flat, all_pred_flat, zero_division=0)
recall    = recall_score   (all_mask_flat, all_pred_flat, zero_division=0)
f1        = f1_score       (all_mask_flat, all_pred_flat, zero_division=0)
inter     = (all_pred_flat * all_mask_flat).sum()
dice_final= (2*inter+1)/(all_pred_flat.sum()+all_mask_flat.sum()+1)
iou_final = (inter+1)/(all_pred_flat.sum()+all_mask_flat.sum()-inter+1)

print('='*50)
print('FINAL EVALUATION RESULTS')
print('='*50)
print(f'  Dice Score  : {dice_final:.4f}')
print(f'  IoU         : {iou_final:.4f}')
print(f'  Precision   : {precision:.4f}')
print(f'  Recall      : {recall:.4f}')
print(f'  F1 Score    : {f1:.4f}')
print(f'  Threshold   : {best_thresh:.3f}')
print(f'  TTA         : {CFG["tta"]}')
print('='*50)
print(f'  Model       : {CFG["arch"]} + {CFG["encoder"]}')
print(f'  Saved to    : {CFG["save_dir"]}/{CFG["save_name"]}')
print('='*50)